# 1-1. Transformer와 Attention

⏱ **소요시간**: 60분

## 학습 목표
- Scaled Dot-Product Attention을 NumPy로 스크래치 구현한다.
- Multi-Head Attention으로 확장하고, MHA / MQA / GQA의 차이를 설명한다.
- Positional Encoding (Sinusoidal / RoPE / ALiBi) 의 동기와 수식을 이해한다.
- KV Cache가 추론 속도를 높이는 원리를 설명한다.
- HuggingFace `bert-base-multilingual-cased` 를 이용해 한국어 문장의 attention을 시각화한다.

## 핵심 키워드
`Scaled Dot-Product Attention`, `Causal Mask`, `Multi-Head`, `MQA/GQA`, `Sinusoidal/RoPE/ALiBi`, `KV Cache`

## 0. 준비

Transformer는 2017년 Vaswani 외, "Attention Is All You Need" 논문에서 등장했다. 오늘 대부분의 LLM (GPT, Llama, Qwen, Mistral)은 이 구조의 디코더(Decoder-only)를 기반으로 한다.

핵심은 **Self-Attention**: 입력 시퀀스 내에서 각 토큰이 다른 토큰을 얼마나 "참조"할지 계산하는 메커니즘이다.

In [1]:
import numpy as np

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)
print('NumPy version:', np.__version__)

NumPy version: 2.4.4


## 1. Scaled Dot-Product Attention

수식:

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$

- $Q \in \mathbb{R}^{L_q \times d_k}$: query — "내가 뭐를 찾고 있는가"
- $K \in \mathbb{R}^{L_k \times d_k}$: key — "나는 무엇에 대한 정보인가"
- $V \in \mathbb{R}^{L_k \times d_v}$: value — "실제로 전달할 내용"
- $\sqrt{d_k}$ 로 나누는 이유: 내적값의 분산이 $d_k$ 에 비례해 커져 softmax가 포화되는 것을 막기 위함.

In [ ]:
def softmax(x, axis=-1):
    # 오버플로우 방지를 위해 최댓값을 빼는다.
    x_max = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - x_max)
    return e / np.sum(e, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """NumPy로 구현한 Scaled Dot-Product Attention.

    Args:
        Q: (Lq, d_k)
        K: (Lk, d_k)
        V: (Lk, d_v)
        mask: (Lq, Lk) 또는 None. 1 이면 허용, 0 이면 -inf 로 마스킹.

    Returns:
        output: (Lq, d_v)
        attn_weights: (Lq, Lk)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)  # (Lq, Lk)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    attn_weights = softmax(scores, axis=-1)
    output = attn_weights @ V  # (Lq, d_v)
    return output, attn_weights


# 예제: 길이 4, d_k=d_v=8 인 가상 토큰 4개
L, d = 4, 8
Q = np.random.randn(L, d)
K = np.random.randn(L, d)
V = np.random.randn(L, d)

out, attn = scaled_dot_product_attention(Q, K, V)
print('output shape:', out.shape)
print('attention weights (각 행의 합은 1):')
print(attn)
print('row sums:', attn.sum(axis=-1))

### 1.1 Causal Mask (Decoder-only LLM의 핵심)

GPT계열 모델은 "다음 토큰을 예측" 하는 학습을 한다. 따라서 **미래 토큰을 미리 보면 안 된다**. 하삼각 마스크로 막는다.

In [4]:
def causal_mask(L):
    """하삼각 마스크: (L, L), 1 = 정보 전달 허용."""
    return np.tril(np.ones((L, L), dtype=np.int32))


mask = causal_mask(L)
print('causal mask:')
print(mask)

out_c, attn_c = scaled_dot_product_attention(Q, K, V, mask=mask)
print('\nmasked attention weights (상삼각이 0이 되어야 정상):')
print(attn_c)

causal mask:
[[1 0 0 0]
 [1 1 0 0]
 [1 1 1 0]
 [1 1 1 1]]

masked attention weights (상삼각이 0이 되어야 정상):
[[1.    0.    0.    0.   ]
 [0.105 0.895 0.    0.   ]
 [0.222 0.07  0.708 0.   ]
 [0.037 0.695 0.061 0.207]]


첫 번째 토큰은 자기 자신만 보고(100%), 두 번째 토큰은 첫/두 번째만, … 동일한 패턴이 나와야 한다.

## 2. Multi-Head Attention (MHA)

하나의 $d_{model}$ 차원 공간을 $h$ 개의 작은 $(d_{model}/h)$ 차원 서브스페이스로 나눠 독립적인 attention을 수행한 뒤 concat한다. 각 head가 서로 다른 관계를 포착할 수 있다 (구문, 개체, 다음 토큰 위치 등).

In [5]:
class MultiHeadAttention:
    """교육용 최소 구현. 실전에서는 PyTorch/Flash-Attn을 쓴다."""

    def __init__(self, d_model=32, n_heads=4, seed=0):
        assert d_model % n_heads == 0, 'd_model은 n_heads의 배수여야 함'
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        rng = np.random.default_rng(seed)
        # 학습 가능한 선형변환을 난수로 초기화
        self.Wq = rng.standard_normal((d_model, d_model)) * 0.1
        self.Wk = rng.standard_normal((d_model, d_model)) * 0.1
        self.Wv = rng.standard_normal((d_model, d_model)) * 0.1
        self.Wo = rng.standard_normal((d_model, d_model)) * 0.1

    def _split_heads(self, X):
        # (L, d_model) -> (n_heads, L, d_head)
        L = X.shape[0]
        return X.reshape(L, self.n_heads, self.d_head).transpose(1, 0, 2)

    def _merge_heads(self, X):
        # (n_heads, L, d_head) -> (L, d_model)
        return X.transpose(1, 0, 2).reshape(-1, self.d_model)

    def __call__(self, X, mask=None):
        Q = self._split_heads(X @ self.Wq)
        K = self._split_heads(X @ self.Wk)
        V = self._split_heads(X @ self.Wv)

        head_outputs = []
        attn_per_head = []
        for h in range(self.n_heads):
            out_h, attn_h = scaled_dot_product_attention(Q[h], K[h], V[h], mask=mask)
            head_outputs.append(out_h)
            attn_per_head.append(attn_h)

        concat = self._merge_heads(np.stack(head_outputs))  # (L, d_model)
        return concat @ self.Wo, np.stack(attn_per_head)


L, d_model, n_heads = 6, 32, 4
X = np.random.randn(L, d_model)
mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
out, attn_all = mha(X, mask=causal_mask(L))
print('output shape:', out.shape)
print('per-head attention shape:', attn_all.shape, '  # (n_heads, L, L)')

output shape: (6, 32)
per-head attention shape: (4, 6, 6)   # (n_heads, L, L)


### 2.1 MHA vs MQA vs GQA 비교

최근 LLM들은 추론 속도와 메모리를 위해 K/V head 수를 줄인다.

| 구조 | Q head 수 | K/V head 수 | KV Cache 크기 | 대표 모델 |
|---|---|---|---|---|
| **MHA** (Multi-Head Attention) | h | h | 100% | GPT-2, BERT |
| **MQA** (Multi-Query Attention) | h | 1 | 1/h | Falcon, PaLM |
| **GQA** (Grouped-Query Attention) | h | g (1 < g < h) | g/h | Llama2 70B, Llama3, Qwen2.5 |

- **MQA**: 메모리는 작지만 품질 저하 가능.
- **GQA**: 중간 절충 — Llama3 70B는 64 Q head / 8 KV head (g=8) 구조.
- 폐쇄망 환경에서 판단 기준: **같은 GPU 메모리에서 얼마나 긴 context를 담을 수 있는가** — GQA/MQA가 유리.

## 3. Positional Encoding

Self-Attention은 본질적으로 **순서 무관(permutation invariant)** 이다. "고객은 은행에 예금을 맡겼다"와 "은행은 고객에 예금을 맡겼다"를 구분하려면 **위치 정보**를 더해야 한다.

### 3.1 Sinusoidal PE (원조 Transformer)

$$
PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)
$$

하나의 위치가 고유한 벡터로 매핑되고, 두 위치의 차이는 선형 변환으로 나타내질 수 있다.

In [ ]:
import matplotlib.pyplot as plt


def sinusoidal_positional_encoding(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, None]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe


pe = sinusoidal_positional_encoding(max_len=64, d_model=128)
plt.figure(figsize=(8, 4))
plt.imshow(pe, cmap='RdBu', aspect='auto')
plt.xlabel('dimension')
plt.ylabel('position')
plt.title('Sinusoidal Positional Encoding (64 pos x 128 dim)')
plt.colorbar()
plt.tight_layout()
plt.show()

### 3.2 RoPE (Rotary Position Embedding) — Llama, Qwen, Mistral

핵심 아이디어: **위치 정보를 Q/K 벡터에 회전 행렬로 곱한다**.

차원을 2개씩 묶어 복소 평면의 점으로 보고, 각 위치 $m$ 에서 각도 $m\theta_i$ 만큼 회전한다.

$$
\text{RoPE}(q, m)_i = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}
$$

**장점**:
- $Q \cdot K^\top$ 의 결과가 두 위치의 **상대 거리** $m - n$ 에만 의존 → 길이 일반화 우수.
- 학습 시 Q/K에만 적용해 V는 건드리지 않음.

### 3.3 ALiBi (Attention with Linear Biases) — MPT, BLOOM

위치 임베딩 대신 **attention score 에 선형 페널티**를 더한다:

$$
\text{scores}_{ij} = \frac{Q_i K_j^\top}{\sqrt{d_k}} - m \cdot |i - j|
$$

- $m$: head별로 고정된 slope
- 학습 전혀 없이도 길이 외삽(extrapolation) 효과가 좋다는 장점.

## 4. KV Cache

Decoder-only LLM의 생성(generation)은 **autoregressive** 하다. 한 토큰씩 넣고 다음 토큰을 예측한다.

매 스텝마다 전체 시퀀스에 대해 K, V를 다시 계산하면 $O(L^2)$. 하지만 K, V는 과거 토큰에 대해 **바뀔 일이 없으므로 캐싱 가능** → 스텝당 $O(L)$.

```python
# 의사코드 (실제에는 per-layer, per-head로 저장)
kv_cache = {'K': None, 'V': None}
for t in range(max_new_tokens):
    q_t = project_q(hidden_t)          # (1, d)   현재 스텝만
    k_t = project_k(hidden_t)          # (1, d)
    v_t = project_v(hidden_t)          # (1, d)

    # 캐시에 이어붙인다
    kv_cache['K'] = concat(kv_cache['K'], k_t)  # (t+1, d)
    kv_cache['V'] = concat(kv_cache['V'], v_t)

    # 새 스텝 계산에는 q_t 와 전체 K/V만 필요
    out_t = softmax(q_t @ kv_cache['K'].T / sqrt(d)) @ kv_cache['V']
    next_token = sample(out_t)
```

### 메모리 스케일

Llama3-8B 기준: `2 * n_layers * seq_len * n_kv_heads * d_head * 2 bytes (fp16)` 

= 2 × 32 × seq_len × 8 × 128 × 2 ≈ **131 KB / token**

즉 4K context = 0.5 GB, 32K context = 4 GB. GQA가 없었다면 8배.

## 5. 실전: 한국어 BERT로 attention 시각화

지금까지 NumPy로 직접 구현했다. 이제 실제 사전 학습된 모델이 한국어에 어떤 attention 패턴을 보이는지 확인한다.

모델: `bert-base-multilingual-cased` (104개 언어 지원, 한국어 포함)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'bert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

sentence = '전자금융거래는 고객이 은행과 대면하지 않고 자동화된 방식으로 이용하는 거래입니다.'
inputs = tokenizer(sentence, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print('토큰 개수:', len(tokens))
print('토큰:', tokens)

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

# attentions: tuple of (batch, n_heads, L, L) 길이 = n_layers
attentions = outputs.attentions
print('레이어 수:', len(attentions))
print('레이어 0 shape:', attentions[0].shape, '  # (batch=1, n_heads=12, L, L)')

In [ ]:
# 특정 레이어·head 선택해 heatmap으로 시각화
LAYER = 6   # 중간 레이어
HEAD = 3    # 0~11

attn_matrix = attentions[LAYER][0, HEAD].numpy()  # (L, L)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(attn_matrix, cmap='viridis')
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=90)
ax.set_yticklabels(tokens)
ax.set_title(f'Attention: layer={LAYER}, head={HEAD}')
ax.set_xlabel('Key (참조된 토큰)')
ax.set_ylabel('Query (참조하는 토큰)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

<!-- optional -->

### 5.1 여러 head를 한번에 비교

각 head가 서로 다른 역할을 학습했음을 확인한다.

In [ ]:
# <!-- optional -->
LAYER = 6
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for h, ax in enumerate(axes.flat):
    ax.imshow(attentions[LAYER][0, h].numpy(), cmap='viridis')
    ax.set_title(f'head {h}', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'Layer {LAYER} — 12 heads')
plt.tight_layout()
plt.show()

## 과제

1. `scaled_dot_product_attention` 함수를 배치 차원을 지원하도록 일반화하시오. 입력 shape가 `(batch, n_heads, L, d)` 일 때 정상 동작해야 합니다.
2. RoPE 회전 행렬을 NumPy로 구현해 Q, K 벡터에 적용하고, 동일 문장에 대해 사인 PE를 사용한 attention 가중치와 RoPE를 사용한 가중치를 비교하세요.
3. 전자금융감독규정 관련 문장 3개를 직접 골라 BERT에 넣고, 12개 레이어 중 **주어-서술어 관계가 선명한 head**를 찾아보세요.

## 더 읽어보기
- Vaswani et al., [Attention Is All You Need (2017)](https://arxiv.org/abs/1706.03762)
- Su et al., [RoFormer: Enhanced Transformer with Rotary Position Embedding (2021)](https://arxiv.org/abs/2104.09864)
- Ainslie et al., [GQA: Training Generalized Multi-Query Transformer Models (2023)](https://arxiv.org/abs/2305.13245)
- [The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/)
- [langchain-kr · 04-Model — LangChain 모델 사용법](https://github.com/teddylee777/langchain-kr/tree/main/04-Model)